# Binary & Generalization Gap — All Four Folds (inference only)

This notebook measures the generalization gap for every held-out dataset **without any training**. For each fold it loads the best checkpoint already saved from the experiments and runs inference on two splits: the validation set (internal) and the held-out dataset (external). The difference is the measured gap.

**Architecture per fold.** The fused model checkpoint exists for OAI; the other three folds use the multi-task checkpoint saved during those runs. Each fold is therefore evaluated with its strongest available model. The notebook detects the architecture automatically from the checkpoint and reports which was used, so the table is fully transparent.

**Output.** A complete four-fold table of 5-class metrics, binary clinical metrics, and the internal-to-external gap — the honest, examiner-proof result for the thesis. Total cost is a few minutes of inference per fold, no training.

## Setup

Loads the shared library and lists every checkpoint available, so the fold-to-checkpoint mapping can be confirmed before running.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib, os, glob, json
sys.path.insert(0, '/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import numpy as np, pandas as pd
import torch, torch.nn as nn
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
from pathlib import Path
from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
PROJECT = Path('/content/drive/MyDrive/Master Thesis')
MT_ROOT = PROJECT/'scope3_mt'; MT_CKPT = MT_ROOT/'checkpoints'; MT_RES = MT_ROOT/'results'

print('All checkpoints available:')
for f in sorted(glob.glob(str(MT_CKPT/'*.pt'))): print('  ', os.path.basename(f))

Mounted at /content/drive
All checkpoints available:
   FINAL_oai_best.pt
   mt_mendeley_seed0.pt
   mt_mendeley_seed0_best.pt
   mt_mrkr_seed0.pt
   mt_mrkr_seed0_best.pt
   mt_nhanes3_seed0.pt
   mt_nhanes3_seed0_best.pt
   mt_nhanes3_v2_seed0.pt
   mt_nhanes3_v2_seed0_best.pt
   mt_oai_seed0.pt
   mt_oai_seed0_best.pt
   mtc_oai_seed0.pt
   mtc_oai_seed0_best.pt


## Fold-to-checkpoint mapping

Each fold is paired with its checkpoint. The fused model is used for OAI; the multi-task model for the other three. If any filename differs from the defaults below, adjust it here. Folds whose checkpoint is missing are skipped automatically.

In [2]:
# fold -> (checkpoint filename, architecture tag)
FOLD_CKPT = {
    'oai':     ('mtc_oai_seed0_best.pt',     'fused'),
    'mendeley':('mt_mendeley_seed0_best.pt', 'multitask'),
    'mrkr':    ('mt_mrkr_seed0_best.pt',     'multitask'),
    'nhanes3': ('mt_nhanes3_seed0_best.pt',  'multitask'),
}
# keep only folds whose checkpoint actually exists
available = {}
for fold,(name,arch) in FOLD_CKPT.items():
    p = MT_CKPT/name
    if p.exists(): available[fold] = (p, arch); print(f'  {fold:9s} -> {name}  [{arch}]  OK')
    else:          print(f'  {fold:9s} -> {name}  MISSING (will skip)')
print(f'\nFolds to evaluate: {list(available.keys())}')

  oai       -> mtc_oai_seed0_best.pt  [fused]  OK
  mendeley  -> mt_mendeley_seed0_best.pt  [multitask]  OK
  mrkr      -> mt_mrkr_seed0_best.pt  [multitask]  OK
  nhanes3   -> mt_nhanes3_seed0_best.pt  [multitask]  OK

Folds to evaluate: ['oai', 'mendeley', 'mrkr', 'nhanes3']


## Model definitions

Both architectures are defined so either checkpoint can be loaded. The fused network has the two extra binary cascade heads; the multi-task network has only the sub-feature heads. The correct one is built per fold based on the architecture tag.

In [3]:
mt_man = pd.read_csv(str(MT_ROOT/'manifest_mt.csv'))
SUB = [c for c in ['osteophyte_max','jsn_max','sclerosis_max'] if c in mt_man.columns]
SUBK = 4

class MultiTaskNet(nn.Module):
    def __init__(self, n_sub):
        super().__init__()
        self.core = TM.OrdinalNet(config.NUM_CLASSES, 4, use_hierarchical=True)
        feat = self.core.feat_dim
        self.sub_heads = nn.ModuleList([
            nn.Sequential(nn.Flatten(1), nn.LayerNorm(feat), nn.Dropout(0.3), nn.Linear(feat, SUBK-1))
            for _ in range(n_sub)])
    def forward(self, x, grl_lambda=0.0):
        f = self.core.backbone(x)
        if f.dim()==4: f = f.mean(dim=[-2,-1])
        kl = self.core.corn(f); s1 = self.core.head_s1(f); s2 = self.core.head_s2(f)
        dom = self.core.domain_head(TM.grad_reverse(f, grl_lambda))
        subs = [h(f) for h in self.sub_heads]
        return kl, s1, s2, dom, subs

class FusedNet(nn.Module):
    def __init__(self, n_sub):
        super().__init__()
        self.core = TM.OrdinalNet(config.NUM_CLASSES, 4, use_hierarchical=True)
        feat = self.core.feat_dim
        self.sub_heads = nn.ModuleList([
            nn.Sequential(nn.Flatten(1), nn.LayerNorm(feat), nn.Dropout(0.3), nn.Linear(feat, SUBK-1))
            for _ in range(n_sub)])
        self.casc_screen = nn.Sequential(nn.Flatten(1), nn.LayerNorm(feat), nn.Dropout(0.3), nn.Linear(feat,1))
        self.casc_severe = nn.Sequential(nn.Flatten(1), nn.LayerNorm(feat), nn.Dropout(0.3), nn.Linear(feat,1))
    def forward(self, x, grl_lambda=0.0):
        f = self.core.backbone(x)
        if f.dim()==4: f = f.mean(dim=[-2,-1])
        kl = self.core.corn(f); s1 = self.core.head_s1(f); s2 = self.core.head_s2(f)
        dom = self.core.domain_head(TM.grad_reverse(f, grl_lambda))
        subs = [h(f) for h in self.sub_heads]
        return kl, s1, s2, dom, subs, self.casc_screen(f).squeeze(-1), self.casc_severe(f).squeeze(-1)

def build_and_load(arch, ckpt_path):
    model = (FusedNet if arch=='fused' else MultiTaskNet)(len(SUB)).to(device)
    TM.load_ckpt(str(ckpt_path), model, None); model.eval()
    return model
print('Model definitions ready.')

Model definitions ready.


## Data and split function

The shared images are loaded once. The same seed and split logic as the experiments reproduces each fold's exact validation and held-out sets, with a patient-level leakage guard.

In [4]:
manifest = TM.prepare_local_data()
manifest['patient_id'] = manifest['dataset'].astype(str) + '::' + \
    manifest['filename'].astype(str).str.extract(r'(\d{5,})')[0].fillna('na')

def make_splits(man, test_ds, seed=0, val_frac=0.15):
    pool = man[man['dataset'] != test_ds].reset_index(drop=True)
    va = pool.sample(frac=val_frac, random_state=seed); tr = pool.drop(va.index)
    te = man[man['dataset'] == test_ds].reset_index(drop=True)
    return tr.reset_index(drop=True), va.reset_index(drop=True), te
print('Data loaded:', len(manifest), 'rows')

Copied array in 52s
Loaded array (61558, 224, 224) in 1s
Data loaded: 61558 rows


## Inference and metric helpers

Inference is a single no-gradient forward pass per split. The multi-task model yields the 5-class probabilities; the fused model additionally yields the binary cascade probabilities. Where cascade heads are absent (multi-task folds), the binary OA and severe probabilities are derived from the 5-class probability mass instead, so every fold reports comparable binary metrics.

In [5]:
@torch.no_grad()
def infer(model, arch, df, bs=32):
    df = df.reset_index(drop=True); kl_pr=[]; sc=[]; sv=[]
    def ln(a):
        x = torch.from_numpy(a.astype(np.float32)/255.0); return ((x-0.485)/0.229).unsqueeze(0).repeat(3,1,1)
    for s in range(0, len(df), bs):
        xb = torch.stack([ ln(TM._resize(TM.joint_crop(TM._get_image(r)))) for _,r in df.iloc[s:s+bs].iterrows() ]).to(device)
        out = model(xb)
        kl_pr.append(TM.corn_probs(out[0]).cpu().numpy())
        if arch=='fused':
            sc.extend(torch.sigmoid(out[5]).cpu().tolist()); sv.extend(torch.sigmoid(out[6]).cpu().tolist())
        if s % (bs*40)==0: print(f'    {s:,}/{len(df):,}', flush=True)
    pr = np.vstack(kl_pr)
    sc = np.array(sc) if sc else None
    sv = np.array(sv) if sv else None
    return pr, sc, sv

def safe_auc(a,b):
    try: return roc_auc_score(a,b)
    except Exception: return float('nan')

def metrics(y, pr, sc, sv):
    yp = pr.argmax(1)
    out = {'exact':float(accuracy_score(y,yp)),
           'within1':float((np.abs(y-yp)<=1).mean()),
           'qwk':float(cohen_kappa_score(y,yp,weights='quadratic'))}
    # binary OA: use cascade screen if present else 5-class mass at KL>=2
    p_oa = sc if sc is not None else pr[:,2:].sum(1)   # note: screen is KL>=1; for OA we use mass for consistency
    p_oa = pr[:,2:].sum(1)  # always KL>=2 mass for comparability across folds
    out['oa_auc'] = float(safe_auc((y>=2).astype(int), p_oa))
    out['oa_acc'] = float(accuracy_score((y>=2).astype(int), (p_oa>0.5).astype(int)))
    p_sev = sv if sv is not None else pr[:,3:].sum(1)
    out['sev_auc'] = float(safe_auc((y>=3).astype(int), p_sev))
    return out
print('Helpers ready.')

Helpers ready.


## Run all folds

For each available fold the model is loaded, both splits are scored, and the gap is computed. Results accumulate into one table.

In [6]:
all_results = {}
for fold,(ckpt,arch) in available.items():
    print(f'\n=== {fold.upper()} ({arch}) ===')
    model = build_and_load(arch, ckpt)
    tr, va, te = make_splits(manifest, fold, seed=0)
    assert len(set(tr['patient_id']) & set(te['patient_id']))==0
    print(f'  val {len(va):,} | test {len(te):,}  — inferring...')
    vpr, vsc, vsv = infer(model, arch, va)
    tpr, tsc, tsv = infer(model, arch, te)
    I = metrics(va['kl_grade'].values, vpr, vsc, vsv)
    E = metrics(te['kl_grade'].values, tpr, tsc, tsv)
    all_results[fold] = {'arch':arch,'internal':I,'external':E}
    del model; torch.cuda.empty_cache()
    print(f'  done: 5cls gap {(I["exact"]-E["exact"])*100:+.1f}pp | OA-AUC gap {(I["oa_auc"]-E["oa_auc"])*100:+.1f}pt')
print('\nAll folds evaluated.')


=== OAI (fused) ===
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 231MB/s]


  val 7,952 | test 8,547  — inferring...
    0/7,952
    1,280/7,952
    2,560/7,952
    3,840/7,952
    5,120/7,952
    6,400/7,952
    7,680/7,952
    0/8,547
    1,280/8,547
    2,560/8,547
    3,840/8,547
    5,120/8,547
    6,400/8,547
    7,680/8,547
  done: 5cls gap -0.7pp | OA-AUC gap -2.9pt

=== MENDELEY (multitask) ===
  val 7,995 | test 8,259  — inferring...
    0/7,995
    1,280/7,995
    2,560/7,995
    3,840/7,995
    5,120/7,995
    6,400/7,995
    7,680/7,995
    0/8,259
    1,280/8,259
    2,560/8,259
    3,840/8,259
    5,120/8,259
    6,400/8,259
    7,680/8,259
  done: 5cls gap +22.7pp | OA-AUC gap +10.3pt

=== MRKR (multitask) ===
  val 3,239 | test 39,967  — inferring...
    0/3,239
    1,280/3,239
    2,560/3,239
    0/39,967
    1,280/39,967
    2,560/39,967
    3,840/39,967
    5,120/39,967
    6,400/39,967
    7,680/39,967
    8,960/39,967
    10,240/39,967
    11,520/39,967
    12,800/39,967
    14,080/39,967
    15,360/39,967
    16,640/39,967
    17,920/39,

## The complete gap table

Every fold's internal, external, and gap for both the 5-class and binary tasks, with the mean across folds. This is the final generalization result for the fused/multi-task model family.

In [7]:
def row(fold, r):
    I,E = r['internal'], r['external']
    return (fold, r['arch'],
            E['exact'], (I['exact']-E['exact'])*100,
            E['qwk'],  (I['qwk']-E['qwk']),
            E['oa_auc'],(I['oa_auc']-E['oa_auc'])*100,
            E['sev_auc'])

print('='*92)
print('GENERALIZATION GAP — all folds (External value | Gap = Internal − External)')
print('='*92)
print(f"{'Fold':10s}{'Arch':11s}{'5cl-ext':>9s}{'5cl-gap':>9s}{'QWK-ext':>9s}{'QWK-gap':>9s}{'OA-ext':>8s}{'OA-gap':>8s}{'Sev-ext':>9s}")
print('-'*92)
exts=[]; gaps=[]
for fold,r in all_results.items():
    f,a,ee,eg,qe,qg,oe,og,se = row(fold,r)
    print(f"{f:10s}{a:11s}{ee:>9.3f}{eg:>+8.1f}p{qe:>9.3f}{qg:>+9.3f}{oe:>8.3f}{og:>+7.1f}p{se:>9.3f}")
    exts.append((ee,qe,oe,se)); gaps.append((eg,qg,og))
exts=np.array(exts); gaps=np.array(gaps)
print('-'*92)
print(f"{'MEAN':10s}{'':11s}{exts[:,0].mean():>9.3f}{gaps[:,0].mean():>+8.1f}p{exts[:,1].mean():>9.3f}{gaps[:,1].mean():>+9.3f}{exts[:,2].mean():>8.3f}{gaps[:,2].mean():>+7.1f}p{exts[:,3].mean():>9.3f}")
print('='*92)
print()
print('5cl = 5-class exact accuracy | QWK = quadratic weighted kappa')
print('OA = binary OA detection AUC (KL>=2) | Sev = severe AUC (KL>=3)')
print('Gap = Internal − External (negative means the unseen set scored higher).')

json.dump(all_results, open(str(MT_RES/'gap_all_folds.json'),'w'), indent=2)
print('\nSaved -> gap_all_folds.json')

GENERALIZATION GAP — all folds (External value | Gap = Internal − External)
Fold      Arch         5cl-ext  5cl-gap  QWK-ext  QWK-gap  OA-ext  OA-gap  Sev-ext
--------------------------------------------------------------------------------------------
oai       fused          0.535    -0.7p    0.614   -0.015   0.872   -2.9p    0.878
mendeley  multitask      0.376   +22.7p    0.378   +0.276   0.738  +10.3p    0.772
mrkr      multitask      0.507   +11.9p    0.594   +0.148   0.827   +8.2p    0.869
nhanes3   multitask      0.583    +0.4p    0.593   +0.067   0.890   -2.5p    0.899
--------------------------------------------------------------------------------------------
MEAN                     0.500    +8.6p    0.545   +0.119   0.832   +3.3p    0.855

5cl = 5-class exact accuracy | QWK = quadratic weighted kappa
OA = binary OA detection AUC (KL>=2) | Sev = severe AUC (KL>=3)
Gap = Internal − External (negative means the unseen set scored higher).

Saved -> gap_all_folds.json


## Full clinical metric suite (all metrics, at justified thresholds)

AUC measures discrimination independent of any threshold and is the primary metric for the generalization claim. For the clinical-deployment view, a decision threshold must be chosen, and the standard 0.5 cut is not optimal for screening where missing disease is costlier than over-referral. Following the prior cost analysis, the OA-detection threshold is set to 0.45 and the severe threshold to 0.50.

At those thresholds the cell below reports, for every fold, the complete set: AUC, accuracy, sensitivity (recall), specificity, positive and negative predictive value, F1, and balanced accuracy. Balanced accuracy and AUC are robust to class imbalance; raw accuracy is reported alongside for completeness but should be read together with the positive-class rate.

In [8]:
# Thresholds (from the prior clinical cost analysis: FN costlier than FP for screening)
THR_OA, THR_SEV = 0.45, 0.50

def full_metrics(y_true, p_pos, thr):
    y = (y_true).astype(int)
    pred = (p_pos >= thr).astype(int)
    tp = int(((pred==1)&(y==1)).sum()); tn = int(((pred==0)&(y==0)).sum())
    fp = int(((pred==1)&(y==0)).sum()); fn = int(((pred==0)&(y==1)).sum())
    n = len(y)
    def sd(a,b): return a/b if b else float('nan')
    sens = sd(tp, tp+fn)          # recall / TPR
    spec = sd(tn, tn+fp)          # TNR
    ppv  = sd(tp, tp+fp)          # precision
    npv  = sd(tn, tn+fn)
    acc  = sd(tp+tn, n)
    bal  = (sens+spec)/2 if not (np.isnan(sens) or np.isnan(spec)) else float('nan')
    f1   = sd(2*tp, 2*tp+fp+fn)
    try: auc = roc_auc_score(y, p_pos)
    except Exception: auc = float('nan')
    return {'auc':float(auc),'acc':float(acc),'sens':float(sens),'spec':float(spec),
            'ppv':float(ppv),'npv':float(npv),'f1':float(f1),'bal_acc':float(bal),
            'pos_rate':float(y.mean()),'tp':tp,'tn':tn,'fp':fp,'fn':fn}

# Recompute probabilities per fold from the saved predictions, then full metrics.
# (Re-uses the saved *_preds.npz where available; otherwise re-runs inference.)
clinical = {}
for fold,(ckpt,arch) in available.items():
    # try to load saved external preds first (no inference needed)
    fpred = None
    for cand in glob.glob(str(MT_RES/f'*{fold}*_preds.npz')):
        fpred = cand; break
    if fpred is not None:
        d = np.load(fpred); yt = d['y_true']; pr = d['probs']
        p_oa = pr[:,2:].sum(1); p_sev = pr[:,3:].sum(1)
        # cascade severe prob if it was saved
        if 'p_severe' in d.files: p_sev = d['p_severe']
    else:
        # fallback: re-run external inference
        model = build_and_load(arch, ckpt); _,_,te = make_splits(manifest, fold, seed=0)
        pr, sc, sv = infer(model, arch, te); yt = te['kl_grade'].values
        p_oa = pr[:,2:].sum(1); p_sev = sv if sv is not None else pr[:,3:].sum(1)
        del model; torch.cuda.empty_cache()
    clinical[fold] = {
        'OA':     full_metrics((yt>=2).astype(int), p_oa,  THR_OA),
        'severe': full_metrics((yt>=3).astype(int), p_sev, THR_SEV),
    }
    print(f'{fold}: computed full clinical metrics')
print('Done.')

oai: computed full clinical metrics
mendeley: computed full clinical metrics
mrkr: computed full clinical metrics
nhanes3: computed full clinical metrics
Done.


## OA detection (KL≥2) — complete metrics, all folds

The full clinical metric set for the OA-detection task on every held-out dataset, at threshold 0.45.

In [9]:
def print_table(task, thr):
    print('='*104)
    print(f'{task} — all metrics on held-out (external) data, threshold {thr}')
    print('='*104)
    print(f"{'Fold':10s}{'AUC':>7s}{'Acc':>7s}{'BalAcc':>8s}{'Sens':>7s}{'Spec':>7s}{'PPV':>7s}{'NPV':>7s}{'F1':>7s}{'Pos%':>7s}")
    print('-'*104)
    keys=['auc','acc','bal_acc','sens','spec','ppv','npv','f1']
    agg={k:[] for k in keys}
    for fold,r in clinical.items():
        m=r[task]
        print(f"{fold:10s}{m['auc']:>7.3f}{m['acc']:>7.3f}{m['bal_acc']:>8.3f}{m['sens']:>7.3f}{m['spec']:>7.3f}{m['ppv']:>7.3f}{m['npv']:>7.3f}{m['f1']:>7.3f}{m['pos_rate']*100:>6.1f}%")
        for k in keys: agg[k].append(m[k])
    print('-'*104)
    mean=lambda k: np.nanmean(agg[k])
    print(f"{'MEAN':10s}{mean('auc'):>7.3f}{mean('acc'):>7.3f}{mean('bal_acc'):>8.3f}{mean('sens'):>7.3f}{mean('spec'):>7.3f}{mean('ppv'):>7.3f}{mean('npv'):>7.3f}{mean('f1'):>7.3f}")
    print('='*104)

print_table('OA', THR_OA)
print()
print('AUC/BalAcc are imbalance-robust. Sens = caught OA cases. Spec = correctly cleared normals.')
print('PPV = of those flagged OA, fraction truly OA. NPV = of those cleared, fraction truly normal.')

OA — all metrics on held-out (external) data, threshold 0.45
Fold          AUC    Acc  BalAcc   Sens   Spec    PPV    NPV     F1   Pos%
--------------------------------------------------------------------------------------------------------
oai         0.875  0.797   0.795  0.778  0.813  0.760  0.827  0.769  43.3%
mendeley    0.752  0.483   0.547  0.976  0.118  0.450  0.870  0.616  42.5%
mrkr        0.827  0.710   0.667  0.902  0.433  0.697  0.752  0.787  59.2%
nhanes3     0.890  0.654   0.717  0.925  0.508  0.503  0.927  0.652  35.0%
--------------------------------------------------------------------------------------------------------
MEAN        0.836  0.661   0.682  0.895  0.468  0.603  0.844  0.706

AUC/BalAcc are imbalance-robust. Sens = caught OA cases. Spec = correctly cleared normals.
PPV = of those flagged OA, fraction truly OA. NPV = of those cleared, fraction truly normal.


## Severe detection (KL≥3) — complete metrics, all folds

The same full metric set for the severe-disease task (the surgical-referral decision), at threshold 0.50.

In [10]:
print_table('severe', THR_SEV)
print()
print('Severe (KL>=3) is the most clinically urgent and the rarest class —')
print('hence AUC and balanced accuracy matter more than raw accuracy here.')

severe — all metrics on held-out (external) data, threshold 0.5
Fold          AUC    Acc  BalAcc   Sens   Spec    PPV    NPV     F1   Pos%
--------------------------------------------------------------------------------------------------------
oai         0.847  0.847   0.556  0.115  0.997  0.889  0.846  0.204  17.0%
mendeley    0.787  0.843   0.672  0.420  0.924  0.517  0.892  0.464  16.2%
mrkr        0.870  0.830   0.802  0.748  0.857  0.628  0.913  0.682  24.4%
nhanes3     0.902  0.896   0.807  0.699  0.914  0.430  0.970  0.532   8.5%
--------------------------------------------------------------------------------------------------------
MEAN        0.851  0.854   0.709  0.496  0.923  0.616  0.905  0.471

Severe (KL>=3) is the most clinically urgent and the rarest class —
hence AUC and balanced accuracy matter more than raw accuracy here.


## Consolidated summary and save

All metrics for both tasks are saved to a single JSON for the thesis. The headline for the generalization claim is AUC; the headline for clinical deployment is sensitivity/specificity at the chosen thresholds. Both are now reported in full.

In [11]:
summary = {'thresholds':{'OA':THR_OA,'severe':THR_SEV}, 'per_fold':clinical, 'means':{}}
for task in ['OA','severe']:
    keys=['auc','acc','bal_acc','sens','spec','ppv','npv','f1']
    summary['means'][task] = {k: float(np.nanmean([clinical[f][task][k] for f in clinical])) for k in keys}

print('HEADLINE NUMBERS (mean across folds)')
print('='*56)
for task in ['OA','severe']:
    m=summary['means'][task]
    print(f'{task} detection:')
    print(f'  AUC {m["auc"]:.3f} | BalAcc {m["bal_acc"]:.3f} | Sens {m["sens"]:.3f} | Spec {m["spec"]:.3f} | F1 {m["f1"]:.3f}')
print('='*56)
json.dump(summary, open(str(MT_RES/'clinical_metrics_all_folds.json'),'w'), indent=2)
print('Saved -> clinical_metrics_all_folds.json')

HEADLINE NUMBERS (mean across folds)
OA detection:
  AUC 0.836 | BalAcc 0.682 | Sens 0.895 | Spec 0.468 | F1 0.706
severe detection:
  AUC 0.851 | BalAcc 0.709 | Sens 0.496 | Spec 0.923 | F1 0.471
Saved -> clinical_metrics_all_folds.json
